# Точечные фичи и канал-фильтр

Тут признаки для будущего реранкера и отдельный дешёвый канал кандидатов на основе фильтров запроса.

In [1]:
import sys

sys.path.append("..")

from collections import defaultdict

import bm25s
import numpy as np
import pandas as pd

from eval import make_val_split, recall_at_50
from preprocessing import extract_label_value, tokenize

train = pd.read_parquet("../data/train.parquet")
train_part, val_queries, val_relevant = make_val_split(train, val_size=0.1, random_state=42)

print("train_part:", len(train_part), "строк")
print("val_queries:", len(val_queries), "запросов")

train_part: 448040 строк
val_queries: 35447 запросов


## BM25 (только для демонстрации фичей на реальных кандидатах ниже)

In [2]:
BM25_CACHE = "../data/bm25s_index"
retriever = bm25s.BM25.load(BM25_CACHE, load_corpus=False)

corpus_item_ids_bm25 = train.drop_duplicates("item_id")["item_id"].tolist()
val_tokens = val_queries["search_query"].apply(tokenize).tolist()
results, scores = retriever.retrieve(
    val_tokens, corpus=corpus_item_ids_bm25, k=50, show_progress=False, n_threads=0
)
bm25_predictions = {
    cid: list(results[i])
    for i, cid in enumerate(val_queries["context_id"])
}
bm25_recall = recall_at_50(bm25_predictions, val_relevant)
print(f"BM25 Recall@50: {bm25_recall:.4f}")

BM25 Recall@50: 0.1861


## Точечные (структурные) фичи для будущего реранкера

Не каналы генерации кандидатов, а признаки для пары (query, candidate item) - понадобятся, когда будем обучать CatBoost/сравнивать с cross-encoder. Всё посчитано на `train_part` (без утечки из val) там, где это влияет на статистику (популярность, центроид локации).

**Гео-дистанция** - самый сильный найденный сигнал.

In [3]:
train_part = train_part.copy()
train_part["item_latitude"] = train_part["item_latitude"].astype(float)
train_part["item_longitude"] = train_part["item_longitude"].astype(float)

loc_centroid = train_part.groupby("item_location_id")[["item_latitude", "item_longitude"]].median()

item_meta = train.drop_duplicates("item_id").set_index("item_id")[[
    "item_location_id", "item_latitude", "item_longitude",
    "item_price", "item_rating", "item_rating_reviews_count",
    "item_is_phone_hidden", "item_is_message_forbidden",
    "item_infm_params_text",
]].copy()
item_meta["item_latitude"] = item_meta["item_latitude"].astype(float)
item_meta["item_longitude"] = item_meta["item_longitude"].astype(float)
item_meta["item_price"] = item_meta["item_price"].astype(float).replace(-1, np.nan)

item_popularity = train_part["item_id"].value_counts()


def haversine(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


def build_pointwise_features(query_row, item_ids):
    meta = item_meta.reindex(item_ids)

    search_loc = query_row["search_location_id"]
    if search_loc in loc_centroid.index:
        c_lat, c_lon = loc_centroid.loc[search_loc, ["item_latitude", "item_longitude"]]
        geo_distance_km = haversine(c_lat, c_lon, meta["item_latitude"], meta["item_longitude"])
    else:
        geo_distance_km = pd.Series(np.nan, index=item_ids)
    geo_exact_match = (meta["item_location_id"] == search_loc).astype(int)

    q_vid = extract_label_value(query_row["search_infm_params_text"], "Вид услуги")
    q_tip = extract_label_value(query_row["search_infm_params_text"], "Тип услуги")
    item_vid = meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Вид услуги"))
    item_tip = meta["item_infm_params_text"].apply(lambda t: extract_label_value(t, "Тип услуги"))

    features = pd.DataFrame({
        "item_id": item_ids,
        "geo_distance_km": geo_distance_km.to_numpy(),
        "geo_exact_match": geo_exact_match.to_numpy(),
        "vid_uslugi_match": (item_vid == q_vid).astype(int).to_numpy() if q_vid else np.nan,
        "tip_uslugi_match": (item_tip == q_tip).astype(int).to_numpy() if q_tip else np.nan,
        "item_price": meta["item_price"].to_numpy(),
        "item_rating": meta["item_rating"].to_numpy(),
        "item_rating_reviews_count": meta["item_rating_reviews_count"].to_numpy(),
        "item_is_phone_hidden": meta["item_is_phone_hidden"].astype(int).to_numpy(),
        "item_is_message_forbidden": meta["item_is_message_forbidden"].astype(int).to_numpy(),
        "item_popularity": [item_popularity.get(iid, 0) for iid in item_ids],
    })
    return features

Проверим на одном реальном запросе с несколькими кандидатами (BM25 top-10 + истинный item):

In [4]:
sample_query = val_queries.iloc[0]
sample_relevant = val_relevant.get(sample_query["context_id"], [])
sample_candidates = list(dict.fromkeys(
    bm25_predictions.get(sample_query["context_id"], [])[:10] + list(sample_relevant)
))

print("query:", sample_query["search_query"], "| фильтры:", sample_query["search_infm_params_text"][:80])
print("истинный item(ы):", sample_relevant)
print()
feats = build_pointwise_features(sample_query, sample_candidates)
feats["is_relevant"] = feats["item_id"].isin(sample_relevant).astype(int)
feats

query: наращивание ногтей | фильтры: Онлайн-запись Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье
истинный item(ы): {'78c2899bda9f2375'}



,item_id,geo_distance_km,geo_exact_match,vid_uslugi_match,tip_uslugi_match,item_price,item_rating,item_rating_reviews_count,item_is_phone_hidden,item_is_message_forbidden,item_popularity,is_relevant
0,3b107bd05ba8ba30,NaN,0,1,1,2700.0,5.000000,3.0,0,0,4,0
1,eb358996cd133e06,NaN,0,1,1,500.0,4.931034,58.0,1,0,3,0
2,ba0358b46cc24dcd,NaN,0,1,1,1500.0,4.826772,127.0,1,0,3,0
3,417bf8b790865877,NaN,0,1,1,1000.0,4.906977,43.0,1,0,1,0
4,c9df5512858ac05f,NaN,0,1,1,2000.0,4.000000,11.0,1,0,1,0
5,f5752c9b70640f65,NaN,0,1,1,1500.0,4.826772,127.0,1,0,2,0
6,69009adb27b2a7a6,NaN,0,1,1,1000.0,5.000000,39.0,1,0,4,0
7,2f04d6a752a10f07,NaN,0,1,1,500.0,4.714286,7.0,0,0,2,0
8,4fa19e069b95169e,NaN,0,1,1,600.0,5.000000,11.0,0,0,1,0
9,75a5dfb0c51b6b8d,NaN,0,1,1,1500.0,4.800000,20.0,1,0,1,0


## Фильтры как канал кандидатов

Раньше `Вид услуги`/`Тип услуги` использовались только как match-фичи выше. Но это можно использовать и как отдельный ИСТОЧНИК кандидатов. На реальных train-парах `Вид услуги` совпадает с ответом в 98.2% случаев, `Тип услуги` - в 79% - когда фильтр вообще задан (67% запросов, у 33% фильтров нет вообще).

In [5]:
item_infm_all = train.drop_duplicates("item_id").set_index("item_id")["item_infm_params_text"]

item_vid_series = item_infm_all.apply(lambda t: extract_label_value(t, "Вид услуги"))
item_tip_series = item_infm_all.apply(lambda t: extract_label_value(t, "Тип услуги"))


def build_value_lookup(value_series, popularity):
    lookup = defaultdict(list)
    for item_id, value in value_series.items():
        if value:
            lookup[value].append(item_id)
    for value in lookup:
        lookup[value].sort(key=lambda iid: -popularity.get(iid, 0))
    return lookup


vid_lookup = build_value_lookup(item_vid_series, item_popularity)
tip_lookup = build_value_lookup(item_tip_series, item_popularity)


def filter_channel_predictions(query_row):
    tip = extract_label_value(query_row["search_infm_params_text"], "Тип услуги")
    if tip and tip in tip_lookup:
        return tip_lookup[tip][:50]
    vid = extract_label_value(query_row["search_infm_params_text"], "Вид услуги")
    if vid and vid in vid_lookup:
        return vid_lookup[vid][:50]
    return []


filter_predictions = {
    row["context_id"]: filter_channel_predictions(row)
    for _, row in val_queries.iterrows()
}
filter_recall = recall_at_50(filter_predictions, val_relevant)
print(f"Фильтры (Тип/Вид услуги => популярность) Recall@50: {filter_recall:.4f}")

Фильтры (Тип/Вид услуги -> популярность) Recall@50: 0.0082


Слабовато, но ок(